In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Create a specific folder if it doesn't exist, and move into it
import os
project_path = '/content/drive/MyDrive/ColabProjects/CS6886_A4'
os.makedirs(project_path, exist_ok=True)

%cd {project_path}
%ls

/content/drive/MyDrive/ColabProjects/CS6886_A4
'Assignment 4 - Simulating a converted SNN.ipynb'
 bugs.md
 build/
 CODE_OF_CONDUCT.md
 community_tutorials/
 CONTRIBUTING.md
 dist/
 docs/
 EE23B141_CS6886A4.ipynb
 LICENCE-DE
 LICENSE
 LICENSE-CN
 LICENSE-FRA
 LICENSE-HI
 publications.md
 README_cn.md
 README.md
 requirements.txt
 setup.py
 SJ-mnist-cnn_model-sample.pth
 spikingjelly/
 spikingjelly.egg-info/


In [28]:
!python3 setup.py install

/usr/local/lib/python3.13/dist-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
!!

        ********************************************************************************
        Please consider removing the following classifiers in favor of a SPDX license expression:

        License :: Other/Proprietary License

        See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
        ********************************************************************************

!!
  self._finalize_license_expression()
running install
/usr/local/lib/python3.13/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This 

In [29]:
import torch
import torchvision
import torch.nn as nn
import spikingjelly
from spikingjelly.activation_based import ann2snn
from tqdm import tqdm
from spikingjelly.activation_based.ann2snn.examples import cnn_mnist as snn 
from spikingjelly.activation_based.ann2snn.sample_models import mnist_cnn
import numpy as np
import matplotlib.pyplot as plt

In [30]:
!nvidia-smi

Sun Sep 13 15:31:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P0             30W /   70W |     157MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### **Task 1** is installing all prerequisites correctly so that the following imports run smoothly. (10 marks)

Install the correct version of torch if you wish to use CUDA.

Install the spikingjelly using the setup.py file only and not with pip using the command 'python setup.py install'.

In [31]:
device_used = 'cuda' if torch.cuda.is_available() else 'cpu'
# set Hyperparameters
snn.hyperparameters.T = 20
snn.hyperparameters.batch_size = 100

device =  device_used # use 'cpu' if CUDA not available
download_dataset = True # downloads MNIST
dataset_dir = './spikingjelly/datasets/mnist'
download_model = False # downloads a 3 layer CNN classifier

if download_model:
        print('Downloading SJ-mnist-cnn_model-sample.pth...')
        ann2snn.download_url("https://ndownloader.figshare.com/files/34960191", './SJ-mnist-cnn_model-sample.pth')

model = mnist_cnn.CNN().to(device)
model.load_state_dict(torch.load('SJ-mnist-cnn_model-sample.pth', map_location=device))
print(model)

CNN(
  (network): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (8): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (9): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (12): Flatten(start_dim=1, end_dim=-1)
    (13): Linear(in_features=32, out_features=10, bias=True)
  )
)


We see the specification of our CNN, having three Conv blocks with [Conv, BatchNorm2d, ReLU, AvgPool2d] modules.

### **Task 2** is calculating the number of MAC operations in each Conv layer. (10 Marks)


Below we take the first two Conv blocks (first 8 modules) as a backbone and the rest of the model as the head.

We freeze the backbone, convert the head to an SNN and evaluate using snn.main.

In [32]:
# TASK 2: number of mac operations in each convolutional layer
# Starting input size for MNIST
in_H, in_W = 28, 28
for i, layer in enumerate(model.network):
    import math
    if isinstance(layer, nn.Conv2d):
        C_in  = layer.in_channels
        C_out = layer.out_channels
        kH, kW = layer.kernel_size   # kernel_size is always stored as a tuple
        sH, sW = layer.stride
        pH, pW = layer.padding

        # out size, since nn.Conv2d doesn't store in or out sizes
        out_H = math.floor((in_H - kH + 2 * pH) / sH) + 1
        out_W = math.floor((in_W - kW + 2 * pW) / sW) + 1

        macs = C_in * kH * kW * out_H * out_W * C_out

        print(f"Layer {i}: Conv2d(Cin={C_in}, Cout={C_out}, k={kH}x{kW}, s={sH}x{sW}, out={out_H}x{out_W})")
        print(f"  MACs: {macs:,}")
        in_H, in_W = out_H, out_W

    elif isinstance(layer, nn.AvgPool2d):
        # AvgPool changes size
        kH = layer.kernel_size
        sH = layer.stride
        pH = layer.padding

        in_H = math.floor((in_H - kH + 2 * pH) / sH) + 1
        in_W = math.floor((in_W - kH + 2 * pH) / sH) + 1
        # (assumes square kernel, which it is here: 2x2)

Layer 0: Conv2d(Cin=1, Cout=32, k=3x3, s=1x1, out=26x26)
  MACs: 194,688
Layer 4: Conv2d(Cin=32, Cout=32, k=3x3, s=1x1, out=11x11)
  MACs: 1,115,136
Layer 8: Conv2d(Cin=32, Cout=32, k=3x3, s=1x1, out=3x3)
  MACs: 82,944


In [34]:
split_index = 8
backbone = model.network[:split_index]
head = model.network[split_index:]
snn_head, vals = snn.main(eval_fn = snn.val, 
    backbone = backbone, 
    head = head, 
    conversion = snn.conversion_job, 
    download_dataset = False,
    device = device_used,
    dataset_dir = dataset_dir)
print(vals)

---------------------------------------------
Converting using 1/4 max(activation) as scales


100%|██████████| 600/600 [00:34<00:00, 17.35it/s]


Simulating...


100%|██████████| 100/100 [00:07<00:00, 13.79it/s]

[0.9837 0.9856 0.9854 0.9862 0.9858 0.9865 0.9864 0.9858 0.986  0.986
 0.9863 0.9865 0.9862 0.986  0.9862 0.9863 0.986  0.9859 0.9861 0.9861]


main() has returned the converted model head and also its accuracy for number of timesteps from 1 to T.

1. We provided a conversion algorithm to snn.main. Go to the file spikingjelly\activation_based\ann2snn\examples\cnn_mnist.py to see how this is defined in snn.conversion_job().
2. You should see some commented out code snippets for other possible conversion jobs. 
3. You can make your own conversion job using any one of those and pass it into snn.main()


### **Task 3** is plotting accuracy vs time steps for all the mentioned conversion jobs in one plot. (10 Marks)
Pick the conversion job that gives best results for you. Stick to this algorithm for all further tasks.

In [36]:
# Task 3: Compare different conversion modes
split_index = 8
backbone = model.network[:split_index]
head = model.network[split_index:]

mode_names = ['max', '99.9%', '1/2 max', '1/4 max']
results = {}

for mode_sel in [0, 1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"Running conversion with mode_sel={mode_sel} ({mode_names[mode_sel]})")
    print('='*60)
    
    snn_head, vals = snn.main(
        eval_fn=snn.val, 
        backbone=backbone, 
        head=head,
        mode_sel=mode_sel,
        conversion=snn.conversion_job, 
        download_dataset=(mode_sel == 0),  # only download on first iteration
        device=device_used,
        dataset_dir=dataset_dir
    )
    results[mode_names[mode_sel]] = vals
    print(f"Final accuracy at T={snn.hyperparameters.T}: {vals[-1]:.4f}")

# Plot accuracy vs timesteps for all modes
plt.figure(figsize=(10, 6))
timesteps = range(1, snn.hyperparameters.T + 1)

for mode_name, vals in results.items():
    plt.plot(timesteps, vals, marker='o', label=mode_name, linewidth=2, markersize=4)

plt.xlabel('Timesteps (T)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy vs Timesteps for Different Conversion Modes', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report best mode
best_mode = max(results.items(), key=lambda x: x[1][-1])
print(f"\nBest conversion mode: {best_mode[0]} with accuracy {best_mode[1][-1]:.4f} at T={snn.hyperparameters.T}")



Running conversion with mode_sel=0 (max)


TypeError: main() got an unexpected keyword argument 'mode_sel'

### **Task 4** is plotting accuracy vs time steps for the original CNN, 1 conv block converted, 2 blocks converted, 3 blocks in one plot. (10 Marks)
Now let us look at the converted model.

In [ ]:
print(snn_head)

Sequential(
  (8): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (11): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (12): Flatten(start_dim=1, end_dim=-1)
  (13): Linear(in_features=32, out_features=10, bias=True)
  (): Module(
    (voltage_hook_0): VoltageHook()
    (spiking0): Module(
      (scaler0): VoltageScaler(0.502180)
      (if_node): IFNode(
        v_threshold=1.0, v_reset=None, detach_reset=False, step_mode=s, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (scaler1): VoltageScaler(1.991318)
    )
  )
)



def forward(self, input):
    input_1 = input
    _8 = getattr(self, "8")(input_1);  input_1 = None
    _spiking0_scaler0 = getattr(self, "").spiking0.scaler0(_8);  _8 = None
    _spiking0_if_node = getattr(self, "").spiking0.if_node(_spiking0_scaler0);  _spiking0_scaler0 = None
    _spiking0_scaler1 = getattr(self, "").spiking0.scaler1(_spiking0_if_node);  _spiking0_if_node = None
    _11 = getattr(self, "11")(_spiking0

SpikingJelly has changed the forward pass of the model significantly. 

In the framework the original Conv2d layer is used but only to simulate the scaling for the inputs to the IF neurons striped across time.
Hence we can use the number of spikes generated multiplied by the fan-out of the layer to estimate the number of accumulate operations (refer https://doi.org/10.3389/fnins.2017.00682)

We use val() in main() after conversion is complete. It evaluates the model on the train dataset and also registers a hook that stores spike counts of a IFNode layer in the model into snn.spike_counts. Dividing this by the size of the dataset provides an average for spike count per layer in a forward pass.

In [ ]:
print(snn.spike_counts)

{'IFNode': 16930766.0}



Go to the file spikingjelly\activation_based\ann2snn\examples\cnn_mnist.py to see how snn.val() is defined

We use val() in main() after conversion is complete. It evaluates the model on the train dataset and also registers a hook that stores spike counts of a IFNode layer in the model into snn.spike_counts.

Write your own custom_val() functions so that you can register hooks to the IFNode() modules for storing spike counts into a dictionary/list for different converted models. (You should need to make minimal changes to val(). Printing a model out might help you figure out how to assign the right hooks.)

### Task 5 is getting average spike count in all layers for a forward pass of a fully converted SNN. (10 Marks)

For a fully converted SNN, use snn.main(custom_val, backbone = None, head = model.network).

**Assume that energy cost of a MAC is 4.6nJ and cost of an accumulate op is 0.6nJ.** Using these values you can estimate the energy savings in converting a layer.

### Task 6 is reporting the best accuracy for an energy savings of 90% or higher. (15 Marks)

### Task 7 is reporting the best energy savings you could achieve from ANN conversion for a loss in accuracy less than 5% from the original CNN. (15 Marks)


Remember, you can scale number of time steps (snn.hyperparameters.T) to scale both computation and accuracy.




So far we have used a really optimistic cost model which only considers energy required for compute operations; we have ignored the costs of writing our compute results to memory.
In reality activation maps are typically too large to fit entirely in CPU cache, especially in vision models. DRAM writes of a 64 byte vector are ~10-50nj and could easily be upwards of 100nj when activating different DRAM rows.

Efficient vision model pipelines try to take maximum advantage of tiling and operator fusion so that computation is focused on smaller regions of the activation map which can fit into cache memory. SRAM writes average around ~1-3nj per vector write across different levels, but this approach introduces a tradeoff between memory traffic and recomputation.

Assume you have a compute pipeline for vision models with high bandwidth memory where performing writes costs 25nj per 64 bytes on average across the available memory hierarchy. Ignore the costs of loading weights, hence only your activation maps contribute to memory-related energy costs. Treat the Conv2d, BatchNorm, ReLU and AvgPool operations within a block as fused for the purpose of memory accounting.

### Task 8 is calculating the memory-related energy costs of each layer in one forward pass of the CNN model. (5 Marks)

Assume these are also the memory-related energy cost of one timestep of running a converted SNN layer. Running the model for multiple time steps incurs this cost at each timestep.

### Task 9 is to report the best energy savings you could achieve from ANN conversion for a loss in accuracy less than 5% with the memory-aware cost model. (15 Marks)

This should highlight the importance of neuromorphic memory innovations like in-memory compute and asynchronous core design.
